# SFINCS — NJ Sandy: forcing phase-lag viewer

The modeled pre-storm tide peaks **late** vs observations, because the northern boundary is
interpolated from the harbor-phase **Battery** gauge (Sandy Hook was excluded — it failed
mid-storm). This notebook A/Bs boundary **forcing sources** to re-phase the coast, keeping the
sealed-premier build/waves fixed so only the forcing changes.

**Status 2026-07-22.** The one arm that exists is **`phaselag_composite`** (`noaa_sandy_composite`
= NOAA harmonic tide + non-tidal residual), scored against the premier **`sealed_faber_waves`**
as control — their `sfincs.inp` are identical key-for-key, so the *only* difference is the
boundary forcing file.

> ⚠️ The earlier three arms (`phaselag_battery` / `_shblend` / `_gtsm`) were **void and deleted**:
> they were staged from the pre-rebuild *leaking* grid. Do not resurrect those labels.

Result: the composite **wins on phase** (Sandy Hook +17.6 → **+7.8** min, Shrewsbury +36.9 →
**+25.5**) but **loses on level** — see §4, and read it before adopting anything.

The estuary leak/Shark-carve story lives in the archived viewer
`archive/notebooks/sfincs-nj-sandy-viz-estuary-leakfix.ipynb`.

## Setup

In [ ]:
# Viz stack (this import also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from hydromt_sfincs import SfincsModel
from nj_sfincs import plots, validate

EXP_ROOT = ROOT / "experiments"
print("experiments dir:", EXP_ROOT)

## 1. Offshore tidal phase by forcing source — the cheap headline (no SFINCS run)

Each source's series nearest the northern anchor (the Sandy Hook gauge) is cross-correlated
against the **real** Sandy Hook observed tide. **Positive = the source is phase-late at the coast**
and will import a late tide (like the Battery); a source near 0 delivers the observed offshore phase.
Sources whose data file isn't built yet show `n/a`.

Early result (built sources): Battery **+21 min**, Sandy Hook blend **0 min**.

In [ ]:
# label -> data_catalog geodataset key
SOURCES = {
    "NOAA Battery (baseline)": "noaa_sandy_nj",
    "NOAA composite (arm)": "noaa_sandy_composite",
    "NOAA Sandy Hook blend": "noaa_sandy_nj_shblend",
    "GTSM-ERA5 total": "gtsm_sandy",
    "GTSM tide-only": "gtsm_sandy_tide",
    "FES2014 tide": "fes_sandy_tide",
}
plots.plot_source_phase(SOURCES);

## 2. Modeled gauge phase — Battery vs blend (vs GTSM)

Each run's label carries its pre-storm phase lag as `Δφ +NN min` (+ = model peaks later than obs).
Read the **left** of the "record ends" line (the pre-storm tide) for the interior gauges. Only runs
already on disk are shown.

In [ ]:
# label -> experiment dir under experiments/ (see nj_sfincs.config EXPERIMENTS).
# CONTROL FIRST: the premier is the baseline, not a deleted phaselag_* arm.
BA = {
    "premier (Battery)": "sealed_faber_waves",
    "composite": "phaselag_composite",
}
BA = {k: v for k, v in BA.items() if (EXP_ROOT / v / "sfincs_map.nc").exists()}
print("runs present:", BA or "(none — expected sealed_faber_waves + phaselag_composite)")
if BA:
    plots.plot_gauge_verification(BA);

## 3. Phase-lag table (minutes; + = model late)

`validate.gauge_phase_lag` per run: Sandy Hook + Shrewsbury from the 10-min his, Shark from the
hourly map at wet channel cells.

Measured (2026-07-22):

| gauge | premier | composite | Δ |
|---|---|---|---|
| Sandy Hook | +17.6 | **+7.8** | −9.8 |
| Shrewsbury | +36.9 | **+25.5** | −11.4 |
| Shark River | +32.8 | +35.1 | +2.3 |

The coastal lag more than halves, and the Shrewsbury interior improves by *the same ~10 min* —
i.e. the interior lag was largely **imported** from the boundary, not generated by conveyance.
But Sandy Hook lands at **+7.8**, not the ~0 that `source_phase_lag` measures at the source: that
residual ~8 min is propagation across the shelf, and it is the honest ceiling of a forcing-only
fix. Shark is unmoved (its lag resolution is hourly, so +2.3 min is inside the noise).

In [ ]:
rows = {}
for label, name in BA.items():
    d = EXP_ROOT / name
    mod = SfincsModel(str(d), data_libs=[str(ROOT / "data" / "data_catalog.yml")], mode="r")
    validate.read_output(mod)  # loads his + map, no floodmap downscale
    rows[label] = validate.gauge_phase_lag(mod, d)
pd.DataFrame(rows).T if rows else print("no runs yet")

## 4. Regression — did re-phasing hurt the crest or the flood extent?

**Yes. The composite is not adoptable as-is.**

| | premier (control) | composite | verdict |
|---|---|---|---|
| Shrewsbury gauge (obs 2.935) | 2.837 (**−0.10**) | 3.186 (**+0.25**) | overshoots |
| HWM bias | +0.318 | **+0.732** | worse |
| HWM RMSE | 0.480 | **0.813** | worse |
| HWM within 0.5 m | **74 %** | **21 %** | much worse |
| MOTF CSI | 0.706 | 0.768 | "better" — but see below |
| MOTF POD / FAR | 0.799 / 0.141 | 0.919 / 0.177 | POD-driven |
| Sandy Hook peak, pre-fail (obs 2.808) | 2.496 (−0.31) | 2.627 (**−0.18**) | better |

**The CSI gain is an artifact, not skill.** POD jumps 0.80 → 0.92 while HWMs say the water is
+0.73 m too high: the model floods *more*, which a wet-heavy extent metric rewards and the
elevation metric convicts. When CSI and HWM disagree, believe HWM — CSI cannot see how deep.

**Why it happened: the composite changed two things, not one.** It re-phased the tide (intended)
*and* it restored Sandy Hook as a **third boundary support point** (2 stations → 3). SFINCS
interpolates linearly along the boundary, so inserting a 3.39 m node between the Battery (3.44 m)
and Atlantic City (1.92 m) lifted the whole mid-coast boundary. At the HWM latitudes that lift is
**+0.20 to +0.23 m**, which accounts for most of the HWM shift in three of four basins:

| basin | Δ boundary level | Δ HWM bias |
|---|---|---|
| atlantic_oceanfront | +0.214 | +0.337 |
| shrewsbury_navesink | +0.222 | +0.321 |
| sandy_hook_bay | +0.234 | +0.258 |
| south_coast | +0.196 | **+0.778** ← nonlinear |

**But the level change is not obviously *wrong*.** The USGS storm-tide sensor at 40.372 °N
observed a **3.465 m** still-water peak; the control's boundary interpolates to 3.042 there and
the composite to 3.264 — the composite is *closer*, and **both are low**. So the open coast wants
*more* water while the HWMs say the model already delivers too much. That contradiction — right
offshore, too high on land — points at over-propagation inland (conveyance / overtopping), which
the +0.2 m lift then amplified nonlinearly (hence south_coast, and hence the POD jump).

**Next arm to disentangle phase from level:** re-run the composite with the **2-station geometry**
(re-phase the Battery and Atlantic City only, no Sandy Hook node). If the phase win survives and
the HWMs return to the control's, phase is free and the level change is the whole cost.

In [ ]:
if BA:
    plots.plot_hwm_residual_panels(BA);
    plots.plot_motf_panels(BA);